# **Factorización por algoritmo de Schnorr de un caso simple**

##### En la siguiente sección se va a mostrar todo el proceso de factorización de un entero N del algoritmo de Schnorr.

In [1]:
%load_ext autoreload
%autoreload 2

from modules import schnorr_lattice as sl
from modules import qaoa
from modules import teoria_numeros as tn
from modules import utils

import numpy as np
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram
from itertools import islice
import time
import csv

import random

## **1. Definición del problema**

El entero que se va a factorizar es $$N = 48567227$$
Queremos construir un retículo de la forma:
$$ B_{n,c} = 
\begin{pmatrix}
f(1) & 0 & \cdots & 0 \\
0 & f(2) & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & f(n) \\
q^c\ln{p_1} & q^c\ln{p_2} & \cdots & q^c\ln{p_n}
\end{pmatrix}
$$


$$
t = 
\begin{pmatrix}
0 \\
\vdots \\
0 \\
q^c\ln{N}
\end{pmatrix} 
\in \mathbb{R}^{n + 1}
$$

- c es un parámetro de precisión
- $f$ es una función que contiene una permutación aleatoria de $(\lceil 1/2 \rceil, \lceil 2/2 \rceil, ..., \lceil n/2 \rceil)$.
- $t$ es el vector objetivo del CVP.
- La dimensión del retículo para que sea sublineal: 
$$n \in O(\frac{\log{N}}{\log{\log{N}}})$$

Entonces: $$n = \frac{l*\log{N}}{\log{\log{N}}}$$
Normalmente tomamos $l \in [ 1, 2 ]$

Por otro lado, en el paper de Yan et al. para conseguir que los vectores encontrados sean $p_B-Smooth$, no tomamos $B = n$. Aumentamos en el tamaño de la base a $B = 2n²$


Para nuestro caso vamos a considerar los siguiente parámetros:
- $N = 48567227$
- $c = 4$
- $l = 1.5$, con esta cantidad nos permite definir un retículo de mayor dimensión lo que puede aumentar la probabilidad de conseguir suficientes $Smooth Pairs$
- $q = 10$
- $seed = 99$

In [2]:
seed = 99

In [3]:
N = 48567227
q = 10
c = 4
l = 1.5

In [4]:
cvp = sl.schnorrCVP(N, c, l, seed)

El numero de bits de N = 48567227 es m = 26
La dimension del reticulo que vamos a tratar es n = 8
La cota smooth que vamos a tomar: 128


Hemos definido con la clase $schnorrCVP$ un objeto que representa el problema. La clase está definicida en el módulo $schnorr\_lattice$ renombrado como $sl$.

En nuestro problema $N = 48567227$ tiene 26 bits. Por tanto, con los parámetros definidos tenemos:
- Lattice Dimension: 8
- Smooth Bound = 128.

El significado del Smooth Bound es que vamos a considerar una base de primos:
$$\{-1, p_1, p_2, ..., p_{128}\}$$
La base tiene un total de SmoothBound + 1 elementos.

## **2. Implementación de la criba mediante la resolución de CVPs**

En esta sección se mostrará todo el proceso implementado.

En nuestro caso como la base $P_{smooth\_bound}$ tiene un total de Smooth Bound + 1 elementos, necesitamos un total de (Smooth Bound + 2) lo denominado Smooth - Relation Pairs para poder construir el sistema de ecuaciones.

Por otro lado, establecemos (de manera arbitraria) el máximo de iteraciones, es decir, la cantidad máxima instancias de CVPs que se va a considerar es de $2^n$ siendo $n$ la dimensión del retículo.

Además, de los bitstrings obtenidos al medir el $Ansatz$ optimizado del $QAOA$ nos quedamos únicamente con los 150 más frecuentes (elegido arbitrariamente).

In [5]:
max_sr_pairs = cvp.smooth_bound + 2 #Tenemos una base de smooth Bound + 1 elementos {-1, p1, p2, ..., pn}
max_iter = 2**cvp.n
elements2sample = 150

En el siguiente código realizamos el bucle donde resolvemos en cada iteración instancias CVP en busca de pares SR.

La información de cada iteración como la cantidad SR-pairs nuevos encontrados, la cantidad de SR-pairs acumulados hasta ahora y el tiempo de ejecución se escribirá en el fichero: 
$$"resolucion\_caso\_simple\_bucle.txt"$$

Por otro lado, los SR-pairs encontrados se guardarán en el fichero:
$$"resolucion\_caso\_simple\_resultado.csv"$$

El formato de cada fila para este fichero será: $$(u, v, u - N*v)$$

##### ***No ejecutar porque alteraría la información de los ficheros .txt y .csv***

In [6]:
""" 
i = 0
found = 0
pairs = set()

with open('resolucion_caso_simple_bucle.txt', 'w', newline = '') as f1, \
    open('resolucion_caso_simple_resultado.csv', 'w', newline = '') as f2:

    pair_writer = csv.writer(f2)
    pair_writer.writerow(['u', 'v', 'u - N*v'])

    f1.write(f"Se va factorizar N = {N}; c (parametro de precision) = {c}; l = 1.5; lattice dimension = {cvp.n}; " + 
             f"smooth bound = {cvp.smooth_bound}; numero de pares = {max_sr_pairs}; max_iteraciones = {max_iter}\n")


    while i < max_iter and found < max_sr_pairs: #Itero para conseguir varias instancias del problema

        #Comienzo de medicion del tiempo de la parte de CVP
        start = time.time_ns()

        instance = cvp.generate_cvp(q, verbose = False)
        cvp_result = cvp.babai_algorithm(instance, delta = 0.75)

        #Fin de medicion del tiempo de la parte de CVP
        stop = time.time_ns()
        babai_ms = (stop - start) / 1_000_000
        

        #Comienzo de medicion del tiempo de Definicion del Problema QUBO
        start = time.time_ns()

        qubo = qaoa.define_qubo(cvp_result.D, cvp_result.res_vector, cvp_result.step_sign, cvp.n)
        Hc, _ = qaoa.define_hamiltonian(qubo)
        circuit = qaoa.construct_circuit(Hc, reps = 1)
        x0 = np.asarray([0.0]*circuit.num_parameters)

        #Fin de medicion del tiempo de Definicion del Problema QUBO
        stop = time.time_ns()
        def_problem_ms = (stop - start) / 1_000_000


        #Comienzo de medicion del tiempo del QAOA
        start = time.time_ns()

        _, optparameters = qaoa.qaoa_algorithm(circuit, Hc, x0)

        #Fin de medicion del tiempo del QAOA
        stop = time.time_ns()
        qaoa_alg_ms = (stop - start) / 1_000_000


        #Comienzo de medicion del tiempo del sampling
        start = time.time_ns()

        results = qaoa.sample_from_parameters(circuit, optparameters, shots = 10_000)

        #Fin de medicion del tiempo del sampling
        stop = time.time_ns()
        samplear_ms = (stop - start) / 1_000_000
        


        #Comienzo de medicion del tiempo de la obtencion de los SR-pairs
        start = time.time_ns()

        resultk = dict(islice(results.items(), elements2sample))
        nD = sl.integer_to_matrix(cvp_result.D)
        vnews = sl.bitstring2latticeVectors(nD, resultk.keys(), cvp_result.step_sign, cvp_result.b_op)
        nB = sl.integer_to_matrix(instance.B)

        uv_pairs = sl.vectors2uv_pairs(nB, vnews, cvp.n)
        sr_pairs = sl.uv_pairs2sr_pairs(uv_pairs, cvp)

        #Fin de medicion del tiempo de la obtencion de los SR-pairs
        stop = time.time_ns()
        get_pairs_ms = (stop - start) / 1_000_000
       


        #Calculo de los pares encontrados
        aux = found
        total = len(sr_pairs)

        for par in sr_pairs:
            if par not in pairs:
                pair_writer.writerow([par[0][0], par[0][1], par[1]])
                pairs.add(par)

        found = len(pairs)
        found_iter = found - aux

        #f1.write(f"Iteracion {i}/{max_iter} encontrado {total} SR Pairs. Encontrados {found_iter} nuevos pares consiguiendo {found}/{max_sr_pairs}. Se ha tardado {time_ms:.3f}ms\n")
        f1.write(f"Iteracion {i}/{max_iter} encontrado {total} SR Pairs. Encontrados {found_iter} nuevos pares consiguiendo {found}/{max_sr_pairs}.\n")
        f1.write(f"Babai tarda: {babai_ms:.3f} ms\n")
        f1.write(f"La definicion del problema tarda: {def_problem_ms:.3f} ms\n")
        f1.write(f"QAOA tarda: {qaoa_alg_ms:.3f} ms\n")
        f1.write(f"Samplear tarda: {samplear_ms:.3f} ms\n")
        f1.write(f"Obtener los SR-Pairs: {get_pairs_ms:.3f} ms\n")
        f1.write(f"Tiempo Total = {babai_ms + def_problem_ms + qaoa_alg_ms + samplear_ms + get_pairs_ms:.3f} ms\n\n")


        i = i + 1  

        del sr_pairs
        del uv_pairs
        del results

del pairs 
"""
        

' \ni = 0\nfound = 0\npairs = set()\n\nwith open(\'resolucion_caso_simple_bucle.txt\', \'w\', newline = \'\') as f1,     open(\'resolucion_caso_simple_resultado.csv\', \'w\', newline = \'\') as f2:\n\n    pair_writer = csv.writer(f2)\n    pair_writer.writerow([\'u\', \'v\', \'u - N*v\'])\n\n    f1.write(f"Se va factorizar N = {N}; c (parametro de precision) = {c}; l = 1.5; lattice dimension = {cvp.n}; " + \n             f"smooth bound = {cvp.smooth_bound}; numero de pares = {max_sr_pairs}; max_iteraciones = {max_iter}\n")\n\n\n    while i < max_iter and found < max_sr_pairs: #Itero para conseguir varias instancias del problema\n\n        #Comienzo de medicion del tiempo de la parte de CVP\n        start = time.time_ns()\n\n        instance = cvp.generate_cvp(q, verbose = False)\n        cvp_result = cvp.babai_algorithm(instance, delta = 0.75)\n\n        #Fin de medicion del tiempo de la parte de CVP\n        stop = time.time_ns()\n        babai_ms = (stop - start) / 1_000_000\n\n\n    

## **3. Resolución de sistema de ecuaciones dado los SR-pairs**

En la siguiente sección se va a utilizar los pares encontrados para resolver los sistemas de ecuaciones.

Tenemos un total $M > B + 1$ pares siendo $B = Smooth Bound$.

$$u_j = \prod_{i = 1}^{B}{p_i^{e_{i,j}}}$$
$$u_j - v_jN = \prod_{i = 0}^{B}{p_i^{e'_{i,j}}}$$

Entonces:
    $$\frac{u_j - v_jN}{u_j} \equiv \prod_{i = 0}^{B}{p_i^{e'_{i,j} - e_{i,j}}} \equiv 1 \mod N$$

Dados $M$ SR-pairs $(u_j, v_j)$, buscamos soluciones para $t_1, t_2, ..., t_M \in \{0, 1\}$ resolviendo mediante la Eliminación Guassiana para Módulo 2 el sistema de ecuaciones:
    $$ \sum_{j = 1}^{M} t_j(e'_{i,j} - e_{i,j}) \equiv 0 \mod 2$$
para $i = 0, ..., n$

### **3.1 Lectura de los SR-pairs**

En la Sección 2 hemos calculado los SR-pairs, por lo que los cargamos del fichero .csv donde los hemos guardado

In [7]:
pairs = set()
us = [] #Conjunto de u's de los pares (u, v)
srs = [] #Conjunto de los valores u - N*v

#Con el fin de no ejecutar el bucle constantemente, cargo los datos del csv
with open('resolucion_caso_simple_resultado.csv', 'r') as f:
    reader = csv.reader(f)

    next(reader)

    for row in reader:
        us.append(int(row[0]))
        srs.append(int(row[2]))


Construimos la matriz de los exponentes para cada par 

In [8]:
matrix = []

for u, sr in zip(us, srs):
    l1 = cvp.get_factors(u)
    l2 = cvp.get_factors(sr)
    
    row = [y - x for x, y in zip(l1, l2)]
    matrix.append(row)
    

### **3.2 Eliminación Gaussiana mod 2**

Dado una matriz $A$ de vectores tal que $A \in \mathbb{M}_{Mx(B + 1)}(\mathbb{F}_2)$.

Es decir, la matriz $A$ es una matriz binaria de $M$ filas y $B$ columnas:

$$
A = 
\begin{pmatrix}
e_1 \\ 
e_2 \\
\vdots \\ 
e_M
\end{pmatrix}
$$
donde cada $e_i \in \mathbb{F_2}^{(B + 1)}$ para $i = 1,...,M$

Buscamos $t = (t_1, t_2, ..., t_M) \in \{0, 1\}$ tal que:

$$
t A = \mathbb{0} \mod 2
$$

En definitiva, buscamos una dependencia lineal entre las filas de $A$ que está garantizada que existe porque $M > (B + 1)$

#### **Algoritmo**

1. Extendemos la matriz A añadiendo a la izquierda la matriz identidad $I_M$: 
    $$A' = [A | I_M]$$
**Propósito: Buscamos convertir $A$ en una matriz con una semi-matriz identidad y en $I_M$, tras las modificaciones a $A$, contiene las combinaciones lineales tales que en $A$ se obtenga fila nula.

2. Inicializamos fila pivote como la fila 0.

3. Por cada columna, buscamos la primera fila $k$ desde el pivote que tenga un $1$. Al encontrarlo lo intercambiamos por la fila pivote:

    $$
    \left[
    \begin{array}{c | c}
    e'_1 & I'_1\\
    e'_2 & I'_2\\
    \vdots & \vdots\\
    e_{pivot} & I_{pivot} \\
    \vdots & \vdots\\
    e_k & I_k \\ 
    \vdots & \vdots\\ 
    e_M & I_M
    \end{array}
    \right]

    \xrightarrow{\mathbb{F_{pivot}} \longleftrightarrow \mathbb{F_k}}

    \left[
    \begin{array}{c | c}
    e'_1 & I'_1\\
    e'_2 & I'_2\\
    \vdots & \vdots\\
    e_{k} & I_{k} \\
    \vdots & \vdots\\
    e_{pivot} & I_{pivot} \\ 
    \vdots & \vdots\\ 
    e_M & I_M
    \end{array}
    \right]
    $$


4. Eliminamos de la columna todos los $1's$ excepto el pivote sumándole la fila pivote. Con estas operaciones vamos obteniendo dependencias poco a poco.

5. Cuando se haya iterado para todas las columnas, en la parte superior tenemos filas con una única columna con el valor 1 y en la parte inferior vectores que sean nulos. Los vectores tal que en la parte de $A$ sean nulos, en la parte de $I_M$ tenemos las combinaciones lineales de los vectores de la matriz original.

    

In [9]:
def eliminacion_gaussiana_mod2(matrix: list[list[int]]) -> list[list[int]]:

    rows = len(matrix)
    cols = len(matrix[0])

    #Construimos la matriz aumentada
    bin_matrix = [[e % 2 for e in vector] + [1 if i == j else 0 for j in range(rows)] for i, vector in enumerate(matrix)]

    
    pivote = 0

    for col in range(cols):
        r = pivote
        while r < rows and bin_matrix[r][col] != 1:
            r += 1
        if r >= rows:
            continue
            
        bin_matrix[pivote], bin_matrix[r] = bin_matrix[r], bin_matrix[pivote]

        #Elimino los 1s de la columna en el resto de las filas
        for r in range(rows):
            if r != pivote and bin_matrix[r][col] == 1:
                bin_matrix[r] = [bin_matrix[pivote][c]^ bin_matrix[r][c] for c in range(cols + rows)]


        pivote += 1


    solutions = []

    for row in bin_matrix:
        left = row[:cols]
        right = row[cols:]
        if all(v == 0 for v in left) and any(v == 1 for v in right):
            solutions.append(right)

    return solutions

In [10]:
solutions = eliminacion_gaussiana_mod2(matrix)
for sol in solutions:
    print(sol)

[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1

### **3.3 Vectores soluciones de los exponentes**

En la seccion anterior obtenemos todas las combinaciones de los vectores (fila) de la matriz $A$.

En esta sección obtenemos los vectores finales mediante el producto matricial:

$$S*A$$

donde $S$ es la matriz donde las filas son las combinaciones lineales.


In [11]:
nmatrix = np.array(matrix)
nsolutions = np.array(solutions)

resultados = np.dot(nsolutions, nmatrix)

In [12]:
for res in resultados:
    print(res)

[  6 -22 -32 -16  -8 -24 -24 -26 -12   2   2   4   4   0   2   2   0   2
   2   2   2   0   0   2   0   0   0   0   0   0   0   0   2   0   0   0
   2   0   0   2   0   0   2   0   2   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   2   0   0   2   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0]
[  6 -22 -32 -16 -42 -16 -36 -24 -22   2   2   4   4   0   2   2   0   2
   2   2   0   0   0   2   2   0   0   0   0   0   4   0   4   0   0   0
   0   0   0   2   0   0   2   2   2   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   2   2   0   0   0   0   2   0   0   2   0   0   2
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0 

### **3.4 Obtener de los valores $X$**

Dado los exponentes de la factorización de $X²$ calculamos $X$

In [13]:
x = []
for sol in resultados:
    aux = cvp.get_valor_by_factors(sol//2)

    x.append(aux)

In [14]:
for f in x: 
    print(f)

1
15228238
33338989
15228238
48567226
15228238
48567226
1
33338989
48567226
1
33338989
15228238
48567226
1
1
15228238
1
33338989
1
1
1
48567226
1
33338989


## **4. Factorizar evaluando los $X's$ encontrados**

**$$N = 48567227 = 6133 * 7919$$**

Entre los valores que encontrados tenemos:
- $X = 33338989$
- $X = 15228238$
- $X = 1$
- $X = 48567226 \equiv {-1} \mod N$

Como indica en el algoritmo original, descartamos los valores de $X = \pm 1 \mod N$

#### **Caso $X = 33338989$**

$$X + 1 = 33338990 = 2 * 5 * 421 * 7919 \mod N \implies gcd(X + 1, N) = 7919$$
$$X - 1 = 33338988 = 2² * 3² * 151 * 6133 \mod N \implies gcd(X - 1, N) = 6133$$

#### **Caso $X = 15228238$**

$$X + 1 = 15228239 = 13 * 191 * 6133 \mod N \implies gcd(X + 1, N) = 6133$$
$$X - 1 = 15228237 =  3 * 641 * 7919 \mod N \implies gcd(X - 1, N) = 7919$$

##### **Factorización satisfactoria**